In [1]:
import pandas as pd
import numpy as np
from faker import Faker
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler, LabelEncoder
from datetime import date

# 1. Generate Synthetic Data with Faker
fake = Faker()
np.random.seed(42)

def generate_cardholder_data(n=500):
    data = []
    for _ in range(n):
        p = fake.profile()
        # Normal range: 500 to 30,000
        balance = np.random.randint(500, 30000)
        age = date.today().year - p['birthdate'].year
        data.append({
            'Name': p['name'], 'Sex': p['sex'], 'Age': age,
            'Occupation': p['job'], 'Balance': balance
        })
    
    df = pd.DataFrame(data)
    
    # Inject Manual Anomalies for Testing
    df.loc[0, 'Balance'] = 250000  # Extreme high balance outlier
    df.loc[1, 'Age'] = 118         # Extreme age outlier
    return df

df = generate_cardholder_data()

# 2. Data Preprocessing
# Encode categorical 'Sex' and scale numerical features
le = LabelEncoder()
df['Sex_Code'] = le.fit_transform(df['Sex'])

features = ['Age', 'Balance', 'Sex_Code']
scaler = StandardScaler()
scaled_data = scaler.fit_transform(df[features])

# 3. Apply KNN Anomaly Detection
# k=5 means we look at the 5 closest neighbors
k = 5
knn = NearestNeighbors(n_neighbors=k)
knn.fit(scaled_data)

# Calculate distances to k-neighbors
distances, _ = knn.kneighbors(scaled_data)

# Average distance is our "Anomaly Score"
df['Anomaly_Score'] = distances.mean(axis=1)

# Define threshold: top 2% of most distant points
threshold = df['Anomaly_Score'].quantile(0.98)
anomalies = df[df['Anomaly_Score'] >= threshold].sort_values('Anomaly_Score', ascending=False)

# 4. Output Anomaly List
print("### DETECTED DATA ANOMALIES (KNN)")
print(f"Threshold Score: {threshold:.2f}\n")
for idx, row in anomalies.head(10).iterrows():
    print(f"* {row['Name']} | Job: {row['Occupation']}")
    print(f"  - Age: {row['Age']} | Balance: ${row['Balance']:,}")
    print(f"  - Distance Score: {row['Anomaly_Score']:.2f} (High distance indicates isolation)")
    print("-" * 30)

### DETECTED DATA ANOMALIES (KNN)
Threshold Score: 0.22

* Andrew Williams | Job: Environmental education officer
  - Age: 23 | Balance: $250,000
  - Distance Score: 13.03 (High distance indicates isolation)
------------------------------
* Mark Bradley | Job: Psychiatrist
  - Age: 2 | Balance: $27,964
  - Distance Score: 0.28 (High distance indicates isolation)
------------------------------
* Jeffrey Acosta | Job: Therapist, music
  - Age: 3 | Balance: $1,002
  - Distance Score: 0.27 (High distance indicates isolation)
------------------------------
* Jeremiah Castro | Job: Chartered management accountant
  - Age: 75 | Balance: $28,799
  - Distance Score: 0.26 (High distance indicates isolation)
------------------------------
* Stanley Brooks | Job: Engineer, drilling
  - Age: 38 | Balance: $1,360
  - Distance Score: 0.23 (High distance indicates isolation)
------------------------------
* Jade Peters MD | Job: Psychologist, educational
  - Age: 62 | Balance: $29,624
  - Distance Sco